# Accel event sim

Thin front-end over the `accel` package (physics lives there; see `accel/sim.py` docstring).
Compares three runs: **ideal TC** (quasi-static, force = min(motor, peak grip) — the old joe-accel model),
**no TC**, and **TC** (slip-target PI in `accel/control.py`).

Legacy version: `accel_weight_transfer.ipynb`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from accel import Car, CurveTire, MotorMap, PacejkaTire, SlipTC, default_tire, quasi_static, simulate

plt.style.use("bmh")
motor = MotorMap("228HV")
tire = default_tire()                  # TTC LC0 Pacejka fit; or CurveTire(mu_peak=1.4)
car = Car()                            # current car: 228HV, COSMX 144s1p, FD 3.0
tc = SlipTC.for_tire(tire)

runs = [quasi_static(car, motor, tire=tire), simulate(car, motor, tire=tire),
        simulate(car, motor, tire=tire, tc=tc)]
pd.DataFrame([r.summary() for r in runs])

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 15), sharex=True)
for r in runs:
    d = r.df
    axes[0].plot(d.t, d.v * 3.6, label=r.label)
    axes[1].plot(d.t, d.a / 9.81, label=r.label)
    axes[2].plot(d.t, d.motor_torque, label=r.label)
    axes[4].plot(d.t, d.bus_v, label=r.label)
for r in runs[1:]:
    axes[3].plot(r.df.t, r.df.slip.clip(upper=1.0), label=r.label)
axes[3].axhline(tc.target_slip, color="r", ls="--", label="TC target")
for ax, y in zip(axes, ["speed (km/h)", "accel (g)", "motor torque (Nm)", "slip ratio", "bus voltage (V)"]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

## Launch detail: what TC is doing

In [ ]:
d = runs[2].df.query("t < 1.0")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(d.t, d.slip, label="measured slip (TC input)")
ax1.plot(d.t, d.kappa, label="tire slip (transient)")
ax1.axhline(tc.target_slip, color="r", ls="--")
ax1.set_ylim(-0.1, 0.6); ax1.legend()
ax2.plot(d.t, d.tc_scale, label="TC torque scale")
ax2.plot(d.t, d.fz_rear / d.fz_rear.max(), label="rear Fz (normalized)")
ax2.legend(); ax2.set_xlabel("time (s)")
plt.tight_layout()

## Final drive sweep

In [ ]:
rows = []
for fd in [2.4, 2.8, 3.2, 3.6, 4.0, 4.5, 5.0]:
    c = Car(final_drive=fd)
    for r in [quasi_static(c, motor, tire=tire), simulate(c, motor, tire=tire),
              simulate(c, motor, tire=tire, tc=tc)]:
        rows.append({"fd": fd, "label": r.label, "time": r.accel_time})
sweep = pd.DataFrame(rows).pivot(index="fd", columns="label", values="time")
ax = sweep.plot(marker="o", figsize=(10, 5), ylabel="accel time (s)", title=f"{motor.name}, {type(tire).__name__}")
sweep

## Validation: KS5e launches

Real launches from `cardata/ks5e` (logger segments, same filter as the old notebook), lined up at torque
application. The logs have no wheel speed or IMU: motor rpm **is** rear wheel speed, so it shows wheelspin
directly. The sim is set up as KS5e (`KS5E_SETUP`: 228HV, 60s8p HE2 pack, FD 3.0 — motor/pack inferred
from the logs, FD assumed). Dashed red is the TC sim's ground speed in motor rpm: the gap to its wheel
line is the slip.

In [ ]:
import math
from accel.compare import KS5E_SETUP, OPTIMUMLAP_SETUP, load_ks5e_runs, load_optimumlap, time_to

ks5e_runs = load_ks5e_runs()
ks5e_car = KS5E_SETUP["car"]
to_rpm = ks5e_car.final_drive / (2 * math.pi * ks5e_car.tire_radius) * 60
ks5e_sims = [
    (simulate(**KS5E_SETUP, tire=CurveTire(), label="no TC, curve tire"), "k"),
    (simulate(**KS5E_SETUP, label="no TC, LC0 tire"), "b"),
    (simulate(**KS5E_SETUP, tc=SlipTC(), label="TC"), "r"),
]

T = 1.5
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for r in ks5e_runs:
    d = r.query("t <= @T")
    ax1.plot(d.t, d.motor_rpm, lw=1, alpha=0.7, label=r.attrs["label"])
    ax2.plot(d.t, d.motor_torque, lw=1, alpha=0.7)
for res, c in ks5e_sims:
    d = res.df.query("t <= @T")
    ax1.plot(d.t, d.motor_rpm, c, lw=2.5, label=f"sim {res.label}")
    ax2.plot(d.t, d.motor_torque, c, lw=2.5, label=f"sim {res.label}")
d = ks5e_sims[-1][0].df.query("t <= @T")
ax1.plot(d.t, d.v * to_rpm, "r--", lw=2, label="sim TC: ground speed")
ax1.set_ylabel("motor rpm (= rear wheel speed)"); ax2.set_ylabel("motor torque (Nm)")
ax2.set_xlabel("time from torque application (s)")
ax1.legend(fontsize=7, ncol=2); ax2.legend(fontsize=8)
ax1.set_title("Launch: KS5e logs vs sim")
plt.tight_layout()

Whole runs (short pulls end where the driver lifted). Real v/x assume no wheelspin, so times are only meaningful for runs that hooked up.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
cols = ["motor_rpm", "motor_torque", "bus_v", "bus_i"]
for r in ks5e_runs:
    for ax, c in zip(axes, cols):
        ax.plot(r.t, r[c], ".", ms=2, alpha=0.6, label=r.attrs["label"])
for res, c in ks5e_sims:
    for ax, col in zip(axes, cols):
        ax.plot(res.df.t, res.df[col], c, lw=2, label=f"sim {res.label}")
for ax, y in zip(axes, ["motor rpm", "motor torque (Nm)", "bus voltage (V)", "bus current (A)"]):
    ax.set_ylabel(y)
axes[0].legend(fontsize=7, ncol=3); axes[-1].set_xlabel("time from torque application (s)")
plt.tight_layout()

pd.DataFrame([{"run": r.attrs["label"], "duration_s": round(r.t.max(), 2),
               "timed_run_s (0.3 -> 75.3 m)": round(time_to(r, 75.3, 0.3), 3)} for r in ks5e_runs]
             + [{"run": f"sim {res.label}", "duration_s": None,
                 "timed_run_s (0.3 -> 75.3 m)": round(res.accel_time, 3)} for res, _ in ks5e_sims])

## Validation: OptimumLap

OptimumLap's 208MV test strip vs the sim with the same setup (`OPTIMUMLAP_SETUP`: 208MV, ratio 4.0).
OptimumLap is a point-mass model with its own grip settings, so this is a cross-check, not ground truth.
It also starts at 3.6 km/h and times from x = 0, so times below are 0 -> 75 m for both.

In [ ]:
olap = load_optimumlap()
olap_sims = [quasi_static(**OPTIMUMLAP_SETUP), simulate(**OPTIMUMLAP_SETUP, tc=SlipTC())]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
axes[0].plot(olap.x, olap.v * 3.6, "k", lw=2, label="OptimumLap")
axes[1].plot(olap.x, olap.a / 9.81, "k", lw=2, label="OptimumLap")
axes[2].plot(olap.x, olap.motor_torque, "k", lw=2, label="OptimumLap (net of its TC)")
for res in olap_sims:
    d = res.df
    axes[0].plot(d.x, d.v * 3.6, label=f"sim {res.label}")
    axes[1].plot(d.x, d.a / 9.81, label=f"sim {res.label}")
    axes[2].plot(d.x, d.motor_torque, label=f"sim {res.label}")
for ax, y in zip(axes, ["speed (km/h)", "accel (g)", "motor torque (Nm)"]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("distance (m)"); axes[-1].set_xlim(0, 75)
plt.tight_layout()

pd.DataFrame([{"run": "OptimumLap", "t_0_to_75m": round(time_to(olap, 75), 3)}]
             + [{"run": f"sim {r.label}", "t_0_to_75m": round(time_to(r.df, 75), 3)} for r in olap_sims])

## Launch strategies

How should torque be applied off the line? Peak motor torque (230 Nm) is above what the rear tires can take
(~199 Nm through launch), so a time-based ramp doesn't help: any ramp that ends at full torque spins eventually.
What matters is the torque *cap*:

- **grip launch map** (`accel.launch.grip_launch_map`): torque vs speed holding the rear tires at a margin below
  peak grip as load transfers. The analytic open-loop optimum; implementable from front wheel speed alone.
- **optimized map** (`optimize_launch_map`): the same thing found numerically, as a check.
- **TC**: stomp the pedal, slip feedback cuts torque.

Each is designed at nominal grip, then run with grip ±10% (track, temperature, tire wear).

In [ ]:
from accel.launch import grip_launch_map, linear_ramp, optimize_launch_map

ideal = quasi_static(car, motor, tire=tire)
launch_runs = [
    simulate(car, motor, tire=tire, label="driver stomp, no TC"),
    simulate(car, motor, tire=tire, torque_request=linear_ramp(motor.peak_torque, 0.5), label="0.5 s ramp, no TC"),
    simulate(car, motor, tire=tire, torque_request=grip_launch_map(car, motor, tire, margin=0.97),
             label="grip map 97%, no TC"),
    simulate(car, motor, tire=tire, tc=SlipTC.for_tire(tire), label="driver stomp + TC"),
]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for r, c in zip([ideal, *launch_runs], ["k", "C0", "C1", "C2", "C3"]):
    d = r.df.query("t < 1.5")
    ax1.plot(d.t, d.motor_torque, c, label=r.label)
    if r is not ideal:
        ax2.plot(d.t, d.slip.clip(upper=0.6), c, label=r.label)
ax1.set_ylabel("motor torque (Nm)"); ax2.set_ylabel("slip ratio (clipped at 0.6)")
ax2.set_xlabel("time (s)"); ax1.legend(); ax2.legend()
plt.tight_layout()
pd.DataFrame([{"strategy": r.label, "accel_time": round(r.accel_time, 3),
               "vs ideal": f"{100 * (r.accel_time / ideal.accel_time - 1):+.2f}%"} for r in [ideal, *launch_runs]])

Robustness: strategies designed at nominal grip, run on a tire with grip scaled 90–110%.

In [ ]:
gmap = grip_launch_map(car, motor, tire, margin=0.97)
strategies = {
    "driver stomp, no TC": dict(),
    "grip map 97%, no TC": dict(torque_request=gmap),
    "driver stomp + TC": dict(tc=True),
    "grip map 97% + TC": dict(torque_request=gmap, tc=True),
}
rows = []
for k in [0.9, 0.95, 1.0, 1.05, 1.1]:
    t_k = PacejkaTire(scale=0.6 * k)
    ideal_k = quasi_static(car, motor, tire=t_k).accel_time
    row = {"grip": f"{k:.0%}", "ideal (s)": round(ideal_k, 3)}
    for name, kw in strategies.items():
        kw = dict(kw)
        tc_k = SlipTC() if kw.pop("tc", False) else None
        r = simulate(car, motor, tire=t_k, tc=tc_k, **kw)
        row[name] = f"{100 * (r.accel_time / ideal_k - 1):+.2f}%"
    rows.append(row)
pd.DataFrame(rows).set_index("grip").T

In [ ]:
# numerical check on the grip map (~1 min). It finds ~0.6% vs ideal at exactly nominal grip, but
# sits on the wheelspin cliff: +0.3 Nm across the map flips it to ~+6%.
RUN_OPTIMIZER = False
if RUN_OPTIMIZER:
    opt_map, opt = optimize_launch_map(car=car, motor=motor, tire=tire)
    g = grip_launch_map(car, motor, tire, margin=0.985)
    print(f"optimized: {opt.accel_time:.3f} s ({100 * (opt.accel_time / ideal.accel_time - 1):+.2f}%)")
    print(pd.DataFrame({"speed (m/s)": opt_map.speeds, "optimized (Nm)": opt_map.torques,
                        "grip map 98.5% (Nm)": [g(0, v) for v in opt_map.speeds]}))

## VCU controller in the loop

`controller/vcu_control.cpp` is firmware-style C++ (no heap, no Arduino, time passed in) that the sim calls through
`accel.firmware`. It only sees what the VCU would: 19-tooth wheel speed sensor edges, motor rpm and VectorNav
frames at their CAN rates with latency and noise, and the driver's torque request. It commands torque every 5 ms,
quantized to 0.1 Nm. Modes: `passthrough`, `slip_pi`, `mu_adaptive` (grip map from an estimated mu, slip PI trim,
mu learned online). `FirmwareController(belief=...)` is the car the controller *thinks* it has.

In [ ]:
from accel.firmware import FirmwareController, Sensors

vcu_runs = {mode: simulate(car, motor, controller=FirmwareController(mode=mode))
            for mode in ["passthrough", "slip_pi", "mu_adaptive"]}
pd.DataFrame([{"controller": m, "accel_time": round(r.accel_time, 3),
               "vs ideal": f"{100 * (r.accel_time / ideal.accel_time - 1):+.2f}%"} for m, r in vcu_runs.items()])

In [ ]:
# what mu_adaptive saw and did, against the truth
r = vcu_runs["mu_adaptive"]
d, log = r.df.query("t < 2"), r.controller_log.query("t < 2")
fig, axes = plt.subplots(4, 1, figsize=(14, 13), sharex=True)
axes[0].plot(d.t, d.v, label="true speed"); axes[0].plot(log.t, log.v_est, label="v_est (VectorNav + wheels)")
axes[0].plot(log.t, log.omega_fl * car.tire_radius, label="front wheel sensor")
axes[1].plot(d.t, d.slip.clip(-0.5, 1), label="true slip"); axes[1].plot(log.t, log.slip.clip(-0.5, 1), label="measured slip")
axes[1].axhline(0.10, color="r", ls="--", label="target")
axes[2].plot(d.t, d.motor_torque, label="motor torque"); axes[2].plot(log.t, log.torque_ff, label="map torque from learned mu")
axes[3].plot(log.t, log.mu, label="mu (learned)"); axes[3].plot(log.t, log.scale, label="PI scale")
for ax, y in zip(axes, ["m/s", "slip ratio", "Nm", ""]):
    ax.set_ylabel(y); ax.legend()
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

### Every coefficient wrong

`accel.uncertainty` draws the simulated car's true grip, mass, CG height, inertias, inverter lag, pitch lag, tire
relaxation, drag, downforce, rolling resistance, driveline efficiency, tire radius and VectorNav accel bias from
ranges (`RANGES`), while every controller keeps its nominal belief. Percent over each plant's own ideal time.

In [ ]:
from accel.launch import grip_launch_map
from accel.uncertainty import RANGES, monte_carlo, sample_plants

N_PLANTS = 20   # ~1 s per plant
gmap = grip_launch_map(car, motor, tire, margin=0.97)
fw = lambda mode: (lambda p: dict(controller=FirmwareController(mode=mode, sensors=Sensors(vn_accel_bias=p.vn_accel_bias))))
strategies = {
    "no TC": lambda p: {},
    "open-loop map 97%": lambda p: dict(torque_request=gmap),
    "Python TC, perfect sensors": lambda p: dict(tc=SlipTC()),
    "VCU slip_pi": fw("slip_pi"),
    "VCU mu_adaptive": fw("mu_adaptive"),
}
mc = monte_carlo(strategies, sample_plants(N_PLANTS, car), motor)
mc[list(strategies)].describe(percentiles=[0.5, 0.9]).T[["mean", "50%", "90%", "max"]].round(2)

In [ ]:
# which wrong coefficients hurt each controller (correlation with % over ideal)
mc[list(RANGES)].corrwith(mc["VCU slip_pi"]).to_frame("slip_pi").join(
    mc[list(RANGES)].corrwith(mc["VCU mu_adaptive"]).to_frame("mu_adaptive")).round(2).sort_values("slip_pi")

## Closing the loop with logs

What a test day looks like, against a hidden "real" car (a random draw from `RANGES`):

1. **Warm-up**: drive gently and coast; the controller calibrates tire radii (front sensors vs VectorNav,
   rear motor vs ground speed while coasting) and VectorNav accel bias (while staged). Learned values persist
   like EEPROM.
2. **Run** with the VCU writing `vcu_log_record_t` records (the binary format in `vcu_control.h`).
3. **Replay** the log through the controller: identical outputs prove the car and sim run the same code.
4. **System ID** (`accel.sysid`) fits grip, mass, tire radius, rear inertia and tire relaxation from the log,
   with error floors measured on random plants, and narrows `RANGES`.
5. **Monte Carlo again** with the narrowed ranges: the prediction tightens around the real car.

In [ ]:
import tempfile, pathlib
from accel.firmware import read_log, replay, warmup, write_log
from accel.sysid import fit_plant

real = sample_plants(1, car, seed=123)[0]                     # the car we don't know
real_sensors = Sensors(vn_accel_bias=real.vn_accel_bias)
sim_real = dict(tire=real.tire, relaxation_length=real.relaxation_length)

learned = warmup(real.car, FirmwareController(sensors=real_sensors), motor, **sim_real)
vcu = FirmwareController(mode="mu_adaptive", sensors=real_sensors, learned=learned)
run = simulate(real.car, motor, controller=vcu, **sim_real)

log_path = pathlib.Path(tempfile.mkdtemp()) / "run.vcu"
write_log(log_path, vcu.records())
inputs, logged = read_log(log_path)
again = replay(inputs, vcu.config(motor), learned=learned, t_offset=vcu.sensors.staged_time)
print(f"log: {len(inputs)} records, {log_path.stat().st_size / 1e3:.0f} kB")
print("replay identical to the run:", (again["torque_cmd"].to_numpy() == run.controller_log["torque_cmd"].to_numpy()).all())
print("learned in warm-up:", {k: round(v, 4) for k, v in learned.items()}, " true radius:", round(real.car.tire_radius, 4))

In [ ]:
fit = fit_plant(inputs, car, motor)
narrowed = fit.ranges(RANGES)
pd.DataFrame([{"coefficient": k, "truth": round(real.draw[k], 4), "fit": round(fit.params[k], 4),
               "std": round(fit.std[k], 4), "prior": RANGES[k], "narrowed": tuple(round(x, 4) for x in narrowed[k])}
              for k in ["grip", "mass", "tire_radius", "relaxation_length"]])

In [ ]:
check = {"VCU mu_adaptive": fw("mu_adaptive")}
rows = []
for name, rg in [("before test day", RANGES), ("after test day", narrowed)]:
    m = monte_carlo(check, sample_plants(15, car, seed=5, ranges=rg), motor)
    rows.append({"ranges": name, "predicted ideal (s)": f"{m.ideal_s.mean():.3f} ± {m.ideal_s.std():.3f}",
                 "mu_adaptive over ideal": f"{m['VCU mu_adaptive'].mean():.2f}%"})
rows.append({"ranges": "the real car", "predicted ideal (s)": f"{quasi_static(real.car, motor, tire=real.tire).accel_time:.3f}",
             "mu_adaptive over ideal": f"{100 * (run.accel_time / quasi_static(real.car, motor, tire=real.tire).accel_time - 1):.2f}%"})
pd.DataFrame(rows)